# 10 Pipeline: generate, screen, rewrite, pool, verify, evaluate

One notebook for the whole candidate-pool pipeline, in two parts:

* **Part A, L4 GPU runtime**: generate train candidates, extract and deduplicate SQL, execute them (correctness and cost), check diversity, rewrite gold for flagged questions, re-check, build the pool, and predict dev and mini-dev.
* **Part B, CPU runtime**: verify the pool's efficiency levels against real timing, evaluate the predictions (EX, R-VES), print the summary.

Start with the **smoke run** (`LIMIT = 20`); after review, run again with `LIMIT = None` and a new `RUN` name.
Every step resumes after a disconnect: just rerun the cells.

**After every code update, reopen this notebook from the GitHub link** (README badge): an already open Colab tab keeps the old cells even though the setup cell pulls the new code.

## Part A: L4 GPU runtime

*Runtime > Change runtime type > L4 GPU.* Then run the cells in order. Installing vLLM takes a few minutes.

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
RUN = "pool_smoke"   # name of this pipeline run; outputs go to 586/effsql/runs/<RUN>/ (use the same value in Part B)
LIMIT = 20           # smoke run: 20 train questions and 20 dev/minidev predictions. Full run: LIMIT = None
LIM = f"--limit {LIMIT}" if LIMIT else ""

!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
print("CPU cores:", os.cpu_count())

### A1. Stage train databases (30 GB, a few minutes) and make sure all train schema caches exist

In [ ]:
!effsql stage --split train
!effsql schema-cache --split train

### A2. Generate: 2 prompts x (1 greedy + 4 samples) per train question

In [ ]:
!effsql generate --split train --run {RUN} {LIM}

### A3. Parse, screen (execute every candidate), diversity check before rewriting

In [ ]:
!effsql parse --split train --run {RUN}
!effsql screen --split train --run {RUN}
!effsql diversity --split train --run {RUN} --round 0

### A4. Rewrite gold for flagged questions, then parse, screen and check diversity again

In [ ]:
!effsql rewrite --run {RUN} --round 1
!effsql parse --split train --run {RUN}
!effsql screen --split train --run {RUN}
!effsql diversity --split train --run {RUN} --round 1

### A5. Build the candidate pool

In [ ]:
!effsql build-pool --split train --run {RUN}

### A6. Zero-shot predictions for dev and mini-dev (greedy, base and efficient prompts)

In [ ]:
for split in ["dev", "minidev"]:
    for prompt in ["base", "efficient"]:
        !effsql predict --split {split} --prompt {prompt} --run {RUN} {LIM}

### Part A done: disconnect now

*Runtime > Disconnect and delete runtime.* An idle L4 keeps using compute units.
Then switch to a CPU runtime (*Runtime > Change runtime type > CPU*) and continue with Part B below.

## Part B: CPU runtime

Timing must run on the CPU runtime, the same machine type as all earlier timing measurements. Use the same `RUN` and `LIMIT` as in Part A.

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
RUN = "pool_smoke"   # name of this pipeline run; outputs go to 586/effsql/runs/<RUN>/ (use the same value in Part B)
LIMIT = 20           # smoke run: 20 train questions and 20 dev/minidev predictions. Full run: LIMIT = None
LIM = f"--limit {LIMIT}" if LIMIT else ""

SAMPLE = LIMIT or 200                                  # questions timed by verify-pool
ONLY = "--only-predicted" if LIMIT else ""             # smoke run: score just the predicted questions
R = f"{DRIVE_ROOT}/effsql/runs/{RUN}"
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

In [ ]:
!effsql stage --split dev

### B1. Verify the pool: does the step-count ordering match real timing?

In [ ]:
!effsql verify-pool --split train --run {RUN} --sample {SAMPLE}

### B2. Evaluate predictions: EX on dev, quick timing on mini-dev

In [ ]:
for prompt in ["base", "efficient"]:
    !effsql evaluate --pred {R}/predictions_dev_{prompt}.jsonl --split dev --timing none --run {RUN}_dev_{prompt} --model {MODEL} --method zero-shot-{prompt} {ONLY}
    !effsql evaluate --pred {R}/predictions_minidev_{prompt}.jsonl --split minidev --timing quick --run {RUN}_minidev_{prompt} --model {MODEL} --method zero-shot-{prompt} {ONLY}

### B3. Official R-VES (reported numbers, full run only)

Exact port of the official procedure: about 1 to 1.5 hours per repeat per prompt. Set `OFFICIAL = True` only for the full run.

In [ ]:
OFFICIAL = False
if OFFICIAL:
    for prompt in ["base", "efficient"]:
        !effsql evaluate --pred {R}/predictions_minidev_{prompt}.jsonl --split minidev --timing official --repeat 3 --run {RUN}_minidev_{prompt}_official --model {MODEL} --method zero-shot-{prompt}

### B4. Summary

In [ ]:
!effsql summary --split train --run {RUN}